# CEMS extractor on Colab (GPU)

Runs the **same** `comment_extractor.py` as the laptop, with `config.colab.yaml`: native annotations first, then coloured-box detection read by **local LightOnOCR**, then (optional) whole-page OCR filtered by the hosted LLM.

Runtime > Change runtime type > **T4 GPU**. Add `OPENROUTER_API_KEY` (and optionally `HF_TOKEN`) in the Secrets panel.

Outputs (`output/`) come back as a zip; ingest them on the laptop with `python ingest.py --input output/master_comments.csv`. Postgres/Qdrant are not used here.

In [ ]:
!nvidia-smi -L || echo 'No GPU: OCR will be very slow'

## 1. Get the repo
Upload `cems_repo.zip` (the repo without `.venv`, `output`, `.env`), or set `GIT_URL`.

In [ ]:
import os, zipfile, pathlib
GIT_URL = ''   # e.g. https://github.com/<you>/cems.git (private repo: use a token in Secrets, not here)
if GIT_URL:
    !git clone -q $GIT_URL /content/cems
else:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    zipfile.ZipFile(name).extractall('/content/cems_unzip')
    root = next(p.parent for p in pathlib.Path('/content/cems_unzip').rglob('comment_extractor.py'))
    !rm -rf /content/cems && cp -r "$root" /content/cems
%cd /content/cems
!ls

## 2. Install

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-ocr.txt
import subprocess, sys
ok = subprocess.run([sys.executable, '-c', 'from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor'], capture_output=True).returncode == 0
if not ok:
    print('Stable transformers lacks LightOnOcr classes -> installing from source (1-2 min)')
    !pip install -q git+https://github.com/huggingface/transformers
!python -c "import transformers; print('transformers', transformers.__version__)"

## 3. Secrets (never typed into a cell that gets saved)

In [ ]:
import os
from getpass import getpass
def secret(name, required=True):
    v = os.getenv(name)
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name)
        except Exception:
            v = None
    if not v and required:
        v = getpass(f'Enter {name}: ')
    if v:
        os.environ[name] = v
secret('OPENROUTER_API_KEY', required=False)  # only needed for the full-page OCR filter / hosted features
secret('HF_TOKEN', required=False)

## 4. PDFs into `input/`

In [ ]:
import pathlib
pathlib.Path('input').mkdir(exist_ok=True)
if not list(pathlib.Path('input').glob('**/*.pdf')):
    from google.colab import files
    for n, d in files.upload().items():
        (pathlib.Path('input') / n).write_bytes(d)
print(len(list(pathlib.Path('input').glob('**/*.pdf'))), 'PDF(s)')

## 5. Dry run (no model, no API), then the real run
`--max-pages 2` for a smoke test; remove it for the full run.

In [ ]:
!python comment_extractor.py --input input --config config.colab.yaml --dry-run

In [ ]:
!python comment_extractor.py --input input --config config.colab.yaml --max-pages 2

## 6. Download outputs

In [ ]:
import shutil
z = shutil.make_archive('/content/cems_output', 'zip', 'output')
from google.colab import files
files.download(z)